# Fragment to 3D Model Converter
## Monocular Depth Estimation & Mesh Generation Pipeline

### Project Overview

This notebook implements an **automated 3D model generation pipeline** that converts 2D image fragments into 3D meshes using **monocular depth estimation**. The project follows these key steps:

| Phase | Description | Technology |
|-------|-------------|------------|
| 1 — Image Loading | Load PNG/JPG images with transparency support | PIL, OpenCV |
| 2 — Depth Estimation | Predict depth maps from single images using AI | MiDaS (Intel) |
| 3 — Mesh Generation | Construct 3D geometry from depth and transparency | NumPy, Trimesh |
| 4 — Texture Mapping | Apply original image as texture to 3D model | Trimesh Materials |
| 5 — Batch Export | Save all models as GLB files for use in 3D viewers | Trimesh Export |

### Technical Approach

**Depth Estimation Method**: Uses **MiDaS Small** (Intel's Monocular Depth Estimation model)
- Pre-trained on diverse datasets (NYU Depth, KITTI, etc.)
- Efficiently estimates relative depth from RGB input
- No stereo pairs or structured light required

**Mesh Construction**: 
- Creates 128×128 vertex grid from image dimensions
- Z-axis displacement: depth × 0.15 (exaggeration factor for visibility)
- Alpha-channel culling: Removes transparent regions (α < 30)
- Generates triangle pairs from grid cells

**Texture Application**:
- Maps original image onto 3D surface via UV coordinates
- Preserves transparency through alpha channel

### Output

- **Format**: GLB (GL Transmission Format, binary)
- **Content**: Geometry + Texture + Metadata
- **Compatibility**: Blender, Three.js, Babylon.js, Unreal Engine

---

## Section 0 — Dependency Installation

Install all required libraries before running the pipeline.

> **Note**: Run this cell once. The first execution of MiDaS will download pre-trained weights (~150 MB).

In [ ]:
!pip install torch torchvision opencv-python trimesh Pillow tqdm

## Section 1 — Setup & Model Initialization

### Configuration

Define input/output paths and initialize the MiDaS depth estimation model.

In [ ]:
import os
import torch
import cv2
import numpy as np
import trimesh
from PIL import Image
from glob import glob
from tqdm import tqdm

# ========== Path Configuration ==========
# Input directory: folder containing 2D image fragments
INPUT_DIR = r"C:\Users\asus\Desktop\RC11 TERM2\data set\mox"

# Output directory: where converted 3D models will be saved
OUTPUT_DIR = r"C:\Users\asus\Desktop\RC11 TERM2\Processed_Models_Mox"

# Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR): 
    os.makedirs(OUTPUT_DIR)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Section 2 — Load Depth Estimation Model

### MiDaS Model

Initialize the Intel MiDaS Small model for monocular depth prediction.

- **Model Size**: Small (faster, suitable for laptops)
- **Inference Speed**: ~0.5-1.0 second per image (on CPU)
- **GPU Acceleration**: Automatically uses CUDA if available

In [ ]:
# Detect GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Initializing Twin City 3D Engine (Device: {device})...")
print()

# Load MiDaS Small model
# This model provides a good balance between speed and accuracy
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.to(device).eval()

# Load corresponding image transform (preprocessing)
transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

print("✓ Model loaded successfully")
print(f"  Device: {device}")
print(f"  Model: MiDaS Small")

## Section 3 — Core Processing Function

### Function: `process_fragment_to_3d()`

Converts a single 2D image fragment to a 3D model through 5 sub-steps:

1. **Image Loading**: Read PNG/JPG with alpha channel
2. **Depth Prediction**: Estimate depth map using MiDaS
3. **Mesh Grid Construction**: Create 128×128 vertex grid
4. **Geometry Generation**: Build triangles, apply depth displacement
5. **Alpha Culling**: Remove transparent vertices (α < 30)
6. **Texture Mapping**: Apply original image as material
7. **Export**: Save as GLB format

In [ ]:
def process_fragment_to_3d(image_path, output_path, z_scale=0.15, mesh_resolution=128, alpha_threshold=30):
    """
    Convert a 2D image fragment to a 3D mesh model.
    
    Args:
        image_path (str): Path to input image (PNG/JPG)
        output_path (str): Path to save output GLB model
        z_scale (float): Depth exaggeration factor (0.15 recommended)
        mesh_resolution (int): Grid resolution (128×128 recommended)
        alpha_threshold (int): Transparency cutoff (0-255)
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # ===== Step 1: Load Image =====
        pil_image = Image.open(image_path).convert("RGBA")
        image_array = np.array(pil_image)
        rgb_image = cv2.cvtColor(image_array, cv2.COLOR_RGBA2RGB)
        image_height, image_width, _ = image_array.shape

        # ===== Step 2: Predict Depth =====
        # Preprocess image for MiDaS
        input_batch = transform(rgb_image).to(device)
        
        # Run inference (disable gradient computation for speed)
        with torch.no_grad():
            depth_prediction = midas(input_batch)
            # Resize to match original image dimensions
            depth_prediction = torch.nn.functional.interpolate(
                depth_prediction.unsqueeze(1), 
                size=(image_height, image_width), 
                mode="bicubic", 
                align_corners=False
            ).squeeze()
        
        # Convert to NumPy and normalize to [0, 1]
        depth_map = depth_prediction.cpu().numpy()
        depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-6)

        # ===== Step 3: Create Mesh Grid =====
        # Generate 2D grid coordinates
        resolution = mesh_resolution
        x_coords = np.linspace(0, 1, resolution)
        y_coords = np.linspace(0, 1, resolution)
        u_grid, v_grid = np.meshgrid(x_coords, y_coords)
        
        # Map grid to image pixel coordinates
        pixel_x = (u_grid * (image_width - 1)).astype(int)
        pixel_y = ((1 - v_grid) * (image_height - 1)).astype(int)

        # ===== Step 4: Generate Geometry =====
        # Sample depth at grid points
        z_heights = depth_map[pixel_y, pixel_x] * z_scale
        
        # Sample alpha channel at grid points
        alpha_channel = image_array[pixel_y, pixel_x, 3]

        # Create vertices: (x - 0.5, z, y - 0.5) normalized coordinates
        vertices = np.stack([
            u_grid.flatten() - 0.5,        # X: centered
            z_heights.flatten(),            # Z: height from depth
            v_grid.flatten() - 0.5          # Y: centered
        ], axis=1)

        # ===== Step 5: Build Triangles with Alpha Culling =====
        alpha_flat = alpha_channel.flatten()
        faces = []
        
        # Iterate through grid cells
        for row in range(resolution - 1):
            for col in range(resolution - 1):
                # Get indices of 4 corners of current cell
                i_top_left = row * resolution + col
                i_top_right = i_top_left + 1
                i_bot_left = (row + 1) * resolution + col
                i_bot_right = i_bot_left + 1
                
                # Only generate triangles if all 4 corners are opaque
                if (alpha_flat[i_top_left] > alpha_threshold and 
                    alpha_flat[i_top_right] > alpha_threshold and
                    alpha_flat[i_bot_left] > alpha_threshold and 
                    alpha_flat[i_bot_right] > alpha_threshold):
                    
                    # Split cell into two triangles
                    faces.append([i_top_left, i_bot_left, i_top_right])
                    faces.append([i_top_right, i_bot_left, i_bot_right])

        # Skip if no valid faces generated
        if not faces: 
            print(f"  ⚠️  No geometry generated for {os.path.basename(image_path)}")
            return False

        # ===== Step 6: Create Mesh Object =====
        mesh = trimesh.Trimesh(
            vertices=vertices, 
            faces=faces, 
            process=False  # Skip auto-cleanup for performance
        )
        
        # ===== Step 7: Apply Texture =====
        # Set UV coordinates (map texture to mesh surface)
        mesh.visual.uv = np.stack([
            u_grid.flatten(), 
            v_grid.flatten()
        ], axis=1)
        
        # Apply original image as texture material
        mesh.visual.material = trimesh.visual.texture.SimpleMaterial(image=pil_image)
        
        # ===== Step 8: Export to GLB =====
        mesh.export(output_path)
        return True
        
    except Exception as error:
        print(f"  ❌ Error processing {os.path.basename(image_path)}: {error}")
        return False

print("✓ Processing function defined")

## Section 4 — Batch Processing Pipeline

### Execution

Scan input directory, process all images, and export 3D models.

**Processing Details**:
- File discovery: PNG + JPG files
- Progress tracking: TQDM progress bar
- Error handling: Skip failed images, continue batch
- Statistics: Count successful conversions

In [ ]:
# ========== Scan for Image Files ==========
image_files = glob(os.path.join(INPUT_DIR, "*.png")) + \
              glob(os.path.join(INPUT_DIR, "*.jpg"))

print(f"📦 Found {len(image_files)} image fragments")
print(f"🔄 Starting batch conversion...")
print()

# ========== Process Each Image ==========
successful_count = 0

for image_path in tqdm(image_files, desc="Converting fragments"):
    # Generate output filename
    base_name = os.path.basename(image_path).split('.')[0]
    output_name = base_name + "_3D.glb"
    output_path = os.path.join(OUTPUT_DIR, output_name)
    
    # Process image
    if process_fragment_to_3d(image_path, output_path):
        successful_count += 1

# ========== Summary Statistics ==========
print()
print("=" * 70)
print(f"✨ Conversion Complete!")
print("=" * 70)
print(f"  Total images: {len(image_files)}")
print(f"  Successful: {successful_count}")
print(f"  Failed: {len(image_files) - successful_count}")
print(f"  Success rate: {100 * successful_count / max(len(image_files), 1):.1f}%")
print(f"📁 Output directory: {OUTPUT_DIR}")
print("=" * 70)

## Section 5 — Verification & Quality Assurance

### Output Inspection

Check generated files and display statistics.

In [ ]:
# List generated GLB files
output_files = glob(os.path.join(OUTPUT_DIR, "*.glb"))

print(f"Generated GLB files: {len(output_files)}")
print()

# Display file sizes
total_size = 0
for glb_file in sorted(output_files)[:10]:  # Show first 10
    file_size_mb = os.path.getsize(glb_file) / (1024 * 1024)
    total_size += file_size_mb
    print(f"  {os.path.basename(glb_file):<50} {file_size_mb:>8.2f} MB")

if len(output_files) > 10:
    print(f"  ... and {len(output_files) - 10} more files")
    # Calculate total size
    total_size = sum(os.path.getsize(f) / (1024 * 1024) for f in output_files)

print()
print(f"Total disk usage: {total_size:.2f} MB")
print()
print("✓ Ready for 3D viewing in Blender, Three.js, or other GLB-compatible viewers")